### What it does
Script to handle words with "-", for example "risk-taker". The aim is to modify it into "risk taker", "risktaker". 
It searchs "-" in words ignoring case. Then it split the words by "-" and joint the words in one taking into account lowcase letter.
It creates new table with the the same columns as original table. Original table with new one are merged and sorted by lemma. 

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'

# pandas library to work with tables/dataframes
import pandas as pd


# Regular expression library
import re
import ast

In [2]:
#file name
file_name = "rules_07SEP_English"

In [3]:
#read csv file to the pandas table
df_rules = pd.read_csv(file_name+".csv")

In [4]:
#get info about the table
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2059 entries, 0 to 2058
Data columns (total 23 columns):
Lemma                     2058 non-null object
Word_Type                 2056 non-null object
Priority                  0 non-null float64
Category                  2059 non-null object
Primary_subcategory       2059 non-null object
Secondary_subcategory     770 non-null object
Alt_Field                 634 non-null object
Alt_Sg_Replacement        1169 non-null object
Identity_first            18 non-null object
Alt_Sg_people_first       182 non-null object
Alt_Sg_reframed           243 non-null object
Alt_Pl_collective_noun    55 non-null object
Alt_Pl                    323 non-null object
Alt_Pl_people_first       115 non-null object
Alt_Pl_reframed           64 non-null object
False_Positives           332 non-null object
3rd_party_alternatives    2 non-null object
3rd_party_source          163 non-null object
Sample_Sentences          6 non-null object
Notes                 

In [5]:
#first 5 row of the table
df_rules.head()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,...,Alt_Pl_people_first,Alt_Pl_reframed,False_Positives,3rd_party_alternatives,3rd_party_source,Sample_Sentences,Notes,ToClarify,Unnamed: 21,Unnamed: 22
0,he,s,NaN,gendered,binary_pronouns,NaN,NaN,they | ((given name)) | ((given name family na...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,"only active if user has toggled on ""singular_t...",NaN,NaN,NaN
1,he or she,s,NaN,gendered,binary_pronouns,NaN,NaN,they | ((given name)) | ((given name family na...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,he/she,s,NaN,gendered,binary_pronouns,NaN,NaN,they | ((given name)) | ((given name family na...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,her,s,NaN,gendered,binary_pronouns,NaN,NaN,them | their | ((given name)) | ((given name f...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,"only active if user has toggled on ""singular_t...",NaN,NaN,NaN
4,hers,s,NaN,gendered,binary_pronouns,NaN,NaN,theirs | ((given name's)),NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Remove unnecessary columns

In [6]:
#create a subset of dataframe from the 'Lemma' till the last column with alternatives '3rd_party_alternatives'
df_rules=df_rules.loc[:, 'Lemma':'3rd_party_alternatives']

In [7]:
#check new table
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2059 entries, 0 to 2058
Data columns (total 17 columns):
Lemma                     2058 non-null object
Word_Type                 2056 non-null object
Priority                  0 non-null float64
Category                  2059 non-null object
Primary_subcategory       2059 non-null object
Secondary_subcategory     770 non-null object
Alt_Field                 634 non-null object
Alt_Sg_Replacement        1169 non-null object
Identity_first            18 non-null object
Alt_Sg_people_first       182 non-null object
Alt_Sg_reframed           243 non-null object
Alt_Pl_collective_noun    55 non-null object
Alt_Pl                    323 non-null object
Alt_Pl_people_first       115 non-null object
Alt_Pl_reframed           64 non-null object
False_Positives           332 non-null object
3rd_party_alternatives    2 non-null object
dtypes: float64(1), object(16)
memory usage: 273.6+ KB


In [8]:
df_rules[df_rules["Lemma"].isna()]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_people_first,Alt_Pl_reframed,False_Positives,3rd_party_alternatives
799,NaN,acr,NaN,style,abbreviation,general,NaN,not applicable | not available,NaN,NaN,NaN,NaN,NaN,NaN,NaN,(N/A),NaN


In [9]:
#convert NaN to string "N/A" in lemmas
df_rules.Lemma.fillna("N/A", inplace=True)

In [41]:
df_rules.iloc[[799]]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_people_first,Alt_Pl_reframed,False_Positives,3rd_party_alternatives
799,N/A,acr,NaN,style,abbreviation,general,NaN,not applicable | not available,NaN,NaN,NaN,NaN,NaN,NaN,NaN,(N/A),NaN


In [11]:
#drop empty lemmas if it is needed
#df_rules = df_rules.dropna(subset = ['Lemma'])

In [12]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2059 entries, 0 to 2058
Data columns (total 17 columns):
Lemma                     2059 non-null object
Word_Type                 2056 non-null object
Priority                  0 non-null float64
Category                  2059 non-null object
Primary_subcategory       2059 non-null object
Secondary_subcategory     770 non-null object
Alt_Field                 634 non-null object
Alt_Sg_Replacement        1169 non-null object
Identity_first            18 non-null object
Alt_Sg_people_first       182 non-null object
Alt_Sg_reframed           243 non-null object
Alt_Pl_collective_noun    55 non-null object
Alt_Pl                    323 non-null object
Alt_Pl_people_first       115 non-null object
Alt_Pl_reframed           64 non-null object
False_Positives           332 non-null object
3rd_party_alternatives    2 non-null object
dtypes: float64(1), object(16)
memory usage: 273.6+ KB


In [13]:
#check fisrts 5 rows of the table
df_rules.head()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_people_first,Alt_Pl_reframed,False_Positives,3rd_party_alternatives
0,he,s,NaN,gendered,binary_pronouns,NaN,NaN,they | ((given name)) | ((given name family na...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,he or she,s,NaN,gendered,binary_pronouns,NaN,NaN,they | ((given name)) | ((given name family na...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,he/she,s,NaN,gendered,binary_pronouns,NaN,NaN,they | ((given name)) | ((given name family na...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,her,s,NaN,gendered,binary_pronouns,NaN,NaN,them | their | ((given name)) | ((given name f...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,hers,s,NaN,gendered,binary_pronouns,NaN,NaN,theirs | ((given name's)),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
#check the number of columns
len(list(df_rules.columns))

17

### Remove white spaces from Category and Primary_subcategory

In [15]:
df_rules['Primary_subcategory'] = df_rules['Primary_subcategory'].str.strip()

In [16]:
df_rules['Category'] = df_rules['Category'].str.strip()

### Convert word_type to lowercase

In [17]:
df_rules['Word_Type'] = df_rules['Word_Type'].str.lower()

### Splitting and merging words by hyphen

In [18]:
list_word = []

list_word_type = []
list_priority = []
list_category = []
list_primary_subcategory = []
list_secondary_subcategory = []
list_alt_field= []
list_alt_sg_replacement = []
list_identity_first = [] 
list_alt_sg_people_first = []
list_alt_sg_reframed = []
list_alt_pl_collective_noun = []
list_alt_pl = []
list_alt_pl_people_first = []
list_alt_pl_reframed = []
list_false_positives = []
list_3rd_party_alternatives = []


for index, row in df_rules.iterrows():
    word = row["Lemma"]
    span = re.search("-", word)
    #only merge and split words if they are two, for example risk-taker (two words: risk and taker)
    if type(span) == re.Match and len(word.rsplit("-"))==2:
        #merge words by "-"
        list_word.append(word.rsplit("-")[0]+word.rsplit("-")[1].lower())
        list_word_type.append(row["Word_Type"])
        list_priority.append(row["Priority"])
        list_category.append(row["Category"])
        list_primary_subcategory.append(row["Primary_subcategory"])
        list_secondary_subcategory.append(row["Secondary_subcategory"])
        list_alt_field.append(row["Alt_Field"])
        list_alt_sg_replacement.append(row["Alt_Sg_Replacement"])
        list_identity_first.append(row["Identity_first"])
        list_alt_sg_people_first.append(row["Alt_Sg_people_first"])
        list_alt_sg_reframed.append(row["Alt_Sg_reframed"])
        list_alt_pl_collective_noun.append(row["Alt_Pl_collective_noun"])
        list_alt_pl.append(row["Alt_Pl"])
        list_alt_pl_people_first.append(row["Alt_Pl_people_first"])
        list_alt_pl_reframed.append(row['Alt_Pl_reframed'])
        list_false_positives.append(row["False_Positives"])
        list_3rd_party_alternatives.append(row["3rd_party_alternatives"])
        
        
        
        #split words by "-"  
        list_word.append(word.rsplit("-")[0]+" "+word.rsplit("-")[1])
        list_word_type.append(row["Word_Type"])
        list_priority.append(row["Priority"])
        list_category.append(row["Category"])
        list_primary_subcategory.append(row["Primary_subcategory"])
        list_secondary_subcategory.append(row["Secondary_subcategory"])
        list_alt_field.append(row["Alt_Field"])
        list_alt_sg_replacement.append(row["Alt_Sg_Replacement"])
        list_identity_first.append(row["Identity_first"])
        list_alt_sg_people_first.append(row["Alt_Sg_people_first"])
        list_alt_sg_reframed.append(row["Alt_Sg_reframed"])
        list_alt_pl_collective_noun.append(row["Alt_Pl_collective_noun"])
        list_alt_pl.append(row["Alt_Pl"])
        list_alt_pl_people_first.append(row["Alt_Pl_people_first"])
        list_alt_pl_reframed.append(row['Alt_Pl_reframed'])
        list_false_positives.append(row["False_Positives"])
        list_3rd_party_alternatives.append(row["3rd_party_alternatives"])
        

### Created new dataframe from lists

In [19]:
#NEW  from the bunch of created lists create a dataframe
data = pd.DataFrame(list(zip(list_word,list_word_type,list_priority, list_category, list_primary_subcategory, list_secondary_subcategory, 
list_alt_field, list_alt_sg_replacement, list_identity_first, list_alt_sg_people_first, list_alt_sg_reframed, list_alt_pl_collective_noun,
list_alt_pl, list_alt_pl_people_first, list_alt_pl_reframed, list_false_positives, list_3rd_party_alternatives)), columns=list(df_rules.columns))

In [20]:
# check number of columns in the new dataframe
len(data.columns)

17

In [21]:
#list of names of the columns
data_columns = list(data.columns)
df_rules_columns = list(df_rules.columns)

### Test name of the columns

In [22]:
for name1, name2 in zip(df_rules_columns, data_columns):
    if name1!=name2:
        print(name1, name2)
    else:
        print("correct")

correct
correct
correct
correct
correct
correct
correct
correct
correct
correct
correct
correct
correct
correct
correct
correct
correct


In [23]:
#check first 5 rows of the new dataframe
data.head()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_people_first,Alt_Pl_reframed,False_Positives,3rd_party_alternatives
0,attentionseeking,a,NaN,gendered,female_stereotype,NaN,NaN,outgoing | at ease in social situations | greg...,NaN,NaN,causing disruption | tending to cause disruption,NaN,NaN,NaN,NaN,NaN,NaN
1,attention seeking,a,NaN,gendered,female_stereotype,NaN,NaN,outgoing | at ease in social situations | greg...,NaN,NaN,causing disruption | tending to cause disruption,NaN,NaN,NaN,NaN,NaN,NaN
2,everyman,s,NaN,gendered,function,NaN,NaN,everybody | everyone | all,NaN,NaN,NaN,NaN,all,NaN,NaN,NaN,NaN
3,every man,s,NaN,gendered,function,NaN,NaN,everybody | everyone | all,NaN,NaN,NaN,NaN,all,NaN,NaN,NaN,NaN
4,gunman,s,NaN,gendered,function,NaN,NaN,shooter | they,NaN,NaN,NaN,NaN,shooters,NaN,NaN,NaN,NaN


In [24]:
#check the last 5 rows of the new dataframe
data.tail()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_people_first,Alt_Pl_reframed,False_Positives,3rd_party_alternatives
339,towel head,s,NaN,openly_discriminating,antimuslim,NaN,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
340,lowclass,a,NaN,unconschious_bias,classism,NaN,NaN,working-class | socio-economically disadvantag...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
341,low class,a,NaN,unconschious_bias,classism,NaN,NaN,working-class | socio-economically disadvantag...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
342,lowerclass,a,NaN,unconschious_bias,classism,NaN,NaN,working-class | socio-economically disadvantag...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
343,lower class,a,NaN,unconschious_bias,classism,NaN,NaN,working-class | socio-economically disadvantag...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### New created dataframe

In [25]:
# check info about new dataframe
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 17 columns):
Lemma                     344 non-null object
Word_Type                 342 non-null object
Priority                  0 non-null float64
Category                  344 non-null object
Primary_subcategory       344 non-null object
Secondary_subcategory     64 non-null object
Alt_Field                 148 non-null object
Alt_Sg_Replacement        162 non-null object
Identity_first            6 non-null object
Alt_Sg_people_first       36 non-null object
Alt_Sg_reframed           84 non-null object
Alt_Pl_collective_noun    10 non-null object
Alt_Pl                    52 non-null object
Alt_Pl_people_first       16 non-null object
Alt_Pl_reframed           12 non-null object
False_Positives           6 non-null object
3rd_party_alternatives    0 non-null float64
dtypes: float64(2), object(15)
memory usage: 45.8+ KB


In [26]:
#original dataframe
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2059 entries, 0 to 2058
Data columns (total 17 columns):
Lemma                     2059 non-null object
Word_Type                 2056 non-null object
Priority                  0 non-null float64
Category                  2059 non-null object
Primary_subcategory       2059 non-null object
Secondary_subcategory     770 non-null object
Alt_Field                 634 non-null object
Alt_Sg_Replacement        1169 non-null object
Identity_first            18 non-null object
Alt_Sg_people_first       182 non-null object
Alt_Sg_reframed           243 non-null object
Alt_Pl_collective_noun    55 non-null object
Alt_Pl                    323 non-null object
Alt_Pl_people_first       115 non-null object
Alt_Pl_reframed           64 non-null object
False_Positives           332 non-null object
3rd_party_alternatives    2 non-null object
dtypes: float64(1), object(16)
memory usage: 273.6+ KB


### Merge two dataframes (original and new one)

In [27]:
#merge two dataframes
df_fin = df_rules.append(data, ignore_index = True)

In [28]:
#check info about merged dataframe
df_fin.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2403 entries, 0 to 2402
Data columns (total 17 columns):
Lemma                     2403 non-null object
Word_Type                 2398 non-null object
Priority                  0 non-null float64
Category                  2403 non-null object
Primary_subcategory       2403 non-null object
Secondary_subcategory     834 non-null object
Alt_Field                 782 non-null object
Alt_Sg_Replacement        1331 non-null object
Identity_first            24 non-null object
Alt_Sg_people_first       218 non-null object
Alt_Sg_reframed           327 non-null object
Alt_Pl_collective_noun    65 non-null object
Alt_Pl                    375 non-null object
Alt_Pl_people_first       131 non-null object
Alt_Pl_reframed           76 non-null object
False_Positives           338 non-null object
3rd_party_alternatives    2 non-null object
dtypes: float64(1), object(16)
memory usage: 319.3+ KB


### Sorted rows in dataframe by lemmas (in alphabetic order)

In [29]:
df_sorted = df_fin.sort_values(by=['Lemma'], inplace=False, ascending=True)

In [30]:
df_sorted.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2403 entries, 626 to 1448
Data columns (total 17 columns):
Lemma                     2403 non-null object
Word_Type                 2398 non-null object
Priority                  0 non-null float64
Category                  2403 non-null object
Primary_subcategory       2403 non-null object
Secondary_subcategory     834 non-null object
Alt_Field                 782 non-null object
Alt_Sg_Replacement        1331 non-null object
Identity_first            24 non-null object
Alt_Sg_people_first       218 non-null object
Alt_Sg_reframed           327 non-null object
Alt_Pl_collective_noun    65 non-null object
Alt_Pl                    375 non-null object
Alt_Pl_people_first       131 non-null object
Alt_Pl_reframed           76 non-null object
False_Positives           338 non-null object
3rd_party_alternatives    2 non-null object
dtypes: float64(1), object(16)
memory usage: 337.9+ KB


### Remove must have from file as false positive

In [31]:
df_sorted[df_sorted["Lemma"]=="must have"]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_people_first,Alt_Pl_reframed,False_Positives,3rd_party_alternatives
2218,must have,a,NaN,style,exaggerating,NaN,NaN,essential | mission-critical | vital | benefic...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2220,must have,s,NaN,style,exaggerating,NaN,NaN,requirement | criterion | essential | required...,NaN,NaN,NaN,NaN,requirements | criteria | essentials | require...,NaN,NaN,NaN,NaN


In [32]:
#get index of false positives
i = df_sorted[((df_sorted.Lemma == 'must have'))].index

In [33]:
#drop rows with false positives
df_sorted = df_sorted.drop(i)

### Remove binary_pronouns  from file

In [34]:
#get index of binary_pronouns
#i_binary = df_sorted[((df_sorted.Primary_subcategory == 'binary_pronouns'))].index

In [35]:
#drop rows with binary pronouns
#df_sorted = df_sorted.drop(i_binary)

### Save the sorted dataframe to csv

In [36]:
#save to csv file after preprocessing
df_sorted.to_csv(file_name+"_h"+".csv", index=False)

In [37]:
#df_sorted["Primary_subcategory"].unique()

In [38]:
#test read of dataframe
df_test = pd.read_csv(file_name+"_h"+".csv")

In [39]:
#check info
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2401 entries, 0 to 2400
Data columns (total 17 columns):
Lemma                     2400 non-null object
Word_Type                 2396 non-null object
Priority                  0 non-null float64
Category                  2401 non-null object
Primary_subcategory       2401 non-null object
Secondary_subcategory     834 non-null object
Alt_Field                 782 non-null object
Alt_Sg_Replacement        1329 non-null object
Identity_first            24 non-null object
Alt_Sg_people_first       218 non-null object
Alt_Sg_reframed           327 non-null object
Alt_Pl_collective_noun    65 non-null object
Alt_Pl                    374 non-null object
Alt_Pl_people_first       131 non-null object
Alt_Pl_reframed           76 non-null object
False_Positives           338 non-null object
3rd_party_alternatives    2 non-null object
dtypes: float64(1), object(16)
memory usage: 319.0+ KB


In [40]:
#check first 5 rows 
df_test.head()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_people_first,Alt_Pl_reframed,False_Positives,3rd_party_alternatives
0,(x)aaS,acr,NaN,style,abbreviation,marketing,NaN,something as a service | something as a servic...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,((x)aaS),NaN
1,A.M.,abbr,NaN,style,abbreviation,job_department_titles,NaN,Master of Arts | Master of Arts (A.M.),NaN,NaN,NaN,NaN,NaN,NaN,NaN,(A.M.),NaN
2,ACCT,abbr,NaN,style,abbreviation,finance,NaN,account,NaN,NaN,NaN,NaN,NaN,NaN,NaN,(ACCT),NaN
3,ADD,abbr,NaN,style,abbreviation,medical,NaN,attention deficit disorder | attention deficit...,NaN,NaN,easily bored | easily distracted | having a po...,NaN,NaN,NaN,NaN,(ADD),NaN
4,ADHD,abbr,NaN,style,abbreviation,medical,NaN,attention deficit hyperactivity disorder | att...,NaN,person with attention deficit hyperactivity di...,fidgety | restless | easily bored,NaN,NaN,people with attention deficit hyperactivity di...,fidgety | restless | easily bored,(ADHD),NaN
